In [ ]:
!pip install pyspark==3.5.0
!pip install spark-nlp==5.5.3

# Introduction

## Corpus et pipeline d'extraction

Le corpus est composé de 14 romans du XIXe siècle répartis en deux mouvements littéraires : le **Romantisme** et le **Réalisme**.
Les textes ont été téléchargés depuis [Project Gutenberg](https://www.gutenberg.org/).

## Configuration de la session Spark

Par défaut, `sparknlp.start()` démarre Spark avec des paramètres opaques :
1 seul core, peu de mémoire, 200 partitions de shuffle.

Nous configurons ici manuellement la session pour :
- Exploiter **tous les cores disponibles** sur Colab (`local[*]`)
- Allouer **8Go de RAM** au driver pour les colonnes NLP lourdes
- Fixer **8 partitions de shuffle** dès le départ, adapté à notre corpus de 14 romans

In [25]:
import sparknlp
import requests
import re
import time
import os
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.ml import Pipeline
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import SentenceDetector, Tokenizer
from pyspark.sql.functions import col, size, udf, explode, lower, count, posexplode, floor
from pyspark.sql.functions import round as spark_round, sum as spark_sum, row_number
from pyspark.sql.window import Window
import plotly.express as px

spark = SparkSession.builder \
    .appName("StyleProject") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.jars.packages", "com.johnsnowlabs.nlp:spark-nlp_2.12:5.5.3") \
    .getOrCreate()

print(f"Spark version : {spark.version}")
print(f"Cores disponibles : {spark.sparkContext.defaultParallelism}")
print(f"shuffle.partitions : {spark.conf.get('spark.sql.shuffle.partitions')}")

Spark version : 3.5.0
Cores disponibles : 8
shuffle.partitions : 8


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = Path("/content/drive/MyDrive/projet_BD/DATA")
RAW_DIR = DATA_DIR / "raw_texts"
RAW_DIR.mkdir(parents=True, exist_ok=True)
print("Drive monté !")

Mounted at /content/drive
Drive monté !


## Téléchargement des textes

Le pipeline ETL télécharge automatiquement les romans depuis Project Gutenberg
et supprime les en-têtes et pieds de page ajoutés par Gutenberg.
Les fichiers sont sauvegardés dans Google Drive dans un dossier
et ne sont pas commités dans le dépôt git.

In [ ]:
DATA_DIR = Path("/content/drive/MyDrive/projet_BD/DATA")
RAW_DIR = DATA_DIR / "raw_texts"
RAW_DIR.mkdir(parents=True, exist_ok=True)

GUTENBERG_URL = "https://www.gutenberg.org/cache/epub/{id}/pg{id}.txt"

CORPUS = [
    {"author": "Balzac",     "title": "Pere_Goriot",          "id": 1237,  "movement": "romantic", "lang": "en"},
    {"author": "Balzac",     "title": "Eugenie_Grandet",       "id": 1608,  "movement": "romantic", "lang": "en"},
    {"author": "Dumas",      "title": "Monte_Cristo",          "id": 17989, "movement": "romantic", "lang": "en"},
    {"author": "Stendhal",   "title": "Rouge_et_Noir",         "id": 798,   "movement": "romantic", "lang": "en"},
    {"author": "Sand",       "title": "Indiana",               "id": 5837,  "movement": "romantic", "lang": "en"},
    {"author": "Bronte_C",   "title": "Jane_Eyre",             "id": 1260,  "movement": "romantic", "lang": "en"},
    {"author": "Bronte_E",   "title": "Wuthering_Heights",     "id": 768,   "movement": "romantic", "lang": "en"},
    {"author": "Scott",      "title": "Ivanhoe",               "id": 82,    "movement": "romantic", "lang": "en"},
    {"author": "Flaubert",   "title": "Madame_Bovary",         "id": 2413,  "movement": "realist",  "lang": "en"},
    {"author": "Zola",       "title": "Germinal",              "id": 5711,  "movement": "realist",  "lang": "en"},
    {"author": "Zola",       "title": "Assommoir",             "id": 8600,  "movement": "realist",  "lang": "en"},
    {"author": "Dickens",    "title": "Great_Expectations",    "id": 1400,  "movement": "realist",  "lang": "en"},
    {"author": "Dickens",    "title": "Oliver_Twist",          "id": 730,   "movement": "realist",  "lang": "en"},
    {"author": "Maupassant", "title": "Bel_Ami",               "id": 4598,  "movement": "realist",  "lang": "en"},
]

def clean_gutenberg(text):
    start = re.search(r"\*\*\* START OF TH(IS|E) PROJECT GUTENBERG", text)
    end   = re.search(r"\*\*\* END OF TH(IS|E) PROJECT GUTENBERG",   text)
    if start and end:
        text = text[start.end():end.start()]
    return text.strip()

def download_all():
    for book in CORPUS:
        path = RAW_DIR / book["author"] / f"{book['title']}.txt"
        path.parent.mkdir(parents=True, exist_ok=True)
        if path.exists():
            print(f"[cache] {book['title']}")
            continue
        url = GUTENBERG_URL.format(id=book["id"])
        response = requests.get(url, timeout=30)
        text = clean_gutenberg(response.text)
        path.write_text(text, encoding="utf-8")
        print(f"[ok] {book['title']} — {len(text):,} caractères")
        time.sleep(1)

download_all()

[cache] Pere_Goriot
[cache] Eugenie_Grandet
[cache] Monte_Cristo
[cache] Rouge_et_Noir
[cache] Indiana
[cache] Jane_Eyre
[cache] Wuthering_Heights
[cache] Ivanhoe
[cache] Madame_Bovary
[cache] Germinal
[cache] Assommoir
[cache] Great_Expectations
[cache] Oliver_Twist
[cache] Bel_Ami


In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, LongType

rows = []
for author_dir in RAW_DIR.iterdir():
    if not author_dir.is_dir():
        continue
    author = author_dir.name
    for txt_file in author_dir.glob("*.txt"):
        title = txt_file.stem
        text = txt_file.read_text(encoding="utf-8")
        movement = next(b["movement"] for b in CORPUS if b["title"] == title)
        lang = next(b["lang"] for b in CORPUS if b["title"] == title)
        rows.append((author, title, movement, lang, text, len(text), len(text.split())))

schema = StructType([
    StructField("author",   StringType(), True),
    StructField("title",    StringType(), True),
    StructField("movement", StringType(), True),
    StructField("lang",     StringType(), True),
    StructField("text",     StringType(), True),
    StructField("n_chars",  LongType(),   True),
    StructField("n_words",  LongType(),   True),
])

df = spark.createDataFrame(rows, schema=schema)
df.select("author", "title", "movement", "lang", "n_chars", "n_words").show()

+----------+------------------+--------+----+-------+-------+
|    author|             title|movement|lang|n_chars|n_words|
+----------+------------------+--------+----+-------+-------+
|    Balzac|       Pere_Goriot|romantic|  en| 571261| 103598|
|    Balzac|   Eugenie_Grandet|romantic|  en| 348752|  66578|
|     Dumas|      Monte_Cristo|romantic|  en| 727540| 121587|
|  Stendhal|     Rouge_et_Noir|romantic|  en|1008461| 171925|
|      Sand|           Indiana|romantic|  en|  79911|  14032|
|  Bronte_C|         Jane_Eyre|romantic|  en|1022418| 185396|
|  Bronte_E| Wuthering_Heights|romantic|  en| 645637| 115949|
|     Scott|           Ivanhoe|romantic|  en|1103759| 191646|
|  Flaubert|     Madame_Bovary| realist|  en| 652156| 115667|
|      Zola|          Germinal| realist|  en|1011885| 164610|
|      Zola|         Assommoir| realist|  en| 870946| 155848|
|   Dickens|Great_Expectations| realist|  en| 994646| 184482|
|   Dickens|      Oliver_Twist| realist|  en| 893187| 157973|
|Maupass

## Sauvegarde en Parquet

Nous sauvegardons le DataFrame au format **Parquet**, un format orienté  colonne qui offre de meilleures performances en lecture que CSV ou JSON, notamment grâce à la compression et au pushdown de prédicats.

In [ ]:
df.write.mode("overwrite").parquet(str(DATA_DIR / "corpus_raw.parquet"))
print("sauvegardé !")

sauvegardé !


## Checkpointing

Après l'annotation NLP, le **lineage** de Spark est très profond : il remonte
jusqu'au téléchargement initial des textes depuis Gutenberg. Si une action
échoue plus tard, Spark doit tout rejouer depuis le début.

Le **checkpoint** règle ce problème en sauvegardant `df_annotated` sur Drive
et en **coupant le lineage**. Toute la stylométrie qui suit repart d'un plan
propre et court.

C'est différent du `cache()` et du `persist()` qui gardent les données en
mémoire mais ne coupent pas le lineage.

In [26]:
#Checkpointing
CHECKPOINT_DIR = str(DATA_DIR / "checkpoints")
spark.sparkContext.setCheckpointDir(CHECKPOINT_DIR)
print(f"Checkpoint dir : {CHECKPOINT_DIR}")

df_annotated = spark.read.parquet(str(DATA_DIR / "corpus_annotated.parquet"))

print("=== Plan AVANT checkpoint ===")
df_annotated.explain()

import time
start = time.perf_counter()
df_annotated_ckpt = df_annotated.checkpoint()
t_ckpt = round(time.perf_counter() - start, 2)
print(f"\nCheckpoint créé en {t_ckpt}s")

print("\n=== Plan APRÈS checkpoint ===")
df_annotated_ckpt.explain()

Checkpoint dir : /content/drive/MyDrive/projet_BD/DATA/checkpoints
=== Plan AVANT checkpoint ===
== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [author#1410,title#1411,movement#1412,lang#1413,text#1414,n_chars#1415L,n_words#1416L,document#1417,sentences#1418,tokens#1419] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/drive/MyDrive/projet_BD/DATA/corpus_annotated.parquet], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<author:string,title:string,movement:string,lang:string,text:string,n_chars:bigint,n_words:...



Checkpoint créé en 3.74s

=== Plan APRÈS checkpoint ===
== Physical Plan ==
*(1) Scan ExistingRDD[author#1410,title#1411,movement#1412,lang#1413,text#1414,n_chars#1415L,n_words#1416L,document#1417,sentences#1418,tokens#1419]




In [27]:
import pandas as pd

pd.DataFrame([
    {
        "Mécanisme":      "cache()",
        "Stockage":       "RAM",
        "Survit à crash": "Non",
        "Coupe lineage":  "Non",
        "Usage typique":  "DataFrame réutilisé plusieurs fois dans la session"
    },
    {
        "Mécanisme":      "persist(DISK_ONLY)",
        "Stockage":       "Disque local",
        "Survit à crash": "Partiel",
        "Coupe lineage":  "Non",
        "Usage typique":  "DataFrame trop lourd pour la RAM"
    },
    {
        "Mécanisme":      "checkpoint()",
        "Stockage":       "Drive / HDFS",
        "Survit à crash": "Oui",
        "Coupe lineage":  "Oui",
        "Usage typique":  "Pipeline long, lineage trop profond"
    },
])

,Mécanisme,Stockage,Survit à crash,Coupe lineage,Usage typique
0,cache(),RAM,Non,Non,DataFrame réutilisé plusieurs fois dans la ses...
1,persist(DISK_ONLY),Disque local,Partiel,Non,DataFrame trop lourd pour la RAM
2,checkpoint(),Drive / HDFS,Oui,Oui,"Pipeline long, lineage trop profond"


### Pourquoi le checkpoint ici ?

Le plan avant checkpoint remonte jusqu'au fichier Parquet annoté,
lui-même produit par le pipeline NLP complet. C'est le point le plus
profond du lineage dans notre projet.

En appliquant le checkpoint après l'annotation, toute la stylométrie
qui suit (FK, Zipf, fenêtres glissantes, dialogue) repart d'un plan
d'exécution court et stable. Le gain de 3.74s à la création est
largement compensé par la robustesse apportée sur l'ensemble du pipeline.

# Annotation avec Spark-NLP

## Pipeline d'annotation

Nous utilisons Spark-NLP 5.5.3 avec PySpark 3.5.0. Le pipeline applique trois étapes successives sur chaque roman en parallèle :

1. **DocumentAssembler** — convertit le texte brut en document Spark-NLP
2. **SentenceDetector** — découpe le document en phrases
3. **Tokenizer** — découpe chaque phrase en mots

In [ ]:
document_assembler = DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

sentence_detector = SentenceDetector() \
    .setInputCols(["document"]) \
    .setOutputCol("sentences")

tokenizer = Tokenizer() \
    .setInputCols(["sentences"]) \
    .setOutputCol("tokens")

pipeline = Pipeline(stages=[
    document_assembler,
    sentence_detector,
    tokenizer
])

model = pipeline.fit(df)
df_annotated = model.transform(df)

df_annotated.write.mode("overwrite").parquet(str(DATA_DIR / "corpus_annotated.parquet"))
print("annoté et sauvegardé !")

annoté et sauvegardé !


In [ ]:
from pyspark.sql.functions import col, size, udf
from pyspark.sql.types import LongType
from pyspark.sql.functions import round as spark_round

def compute_syllables(text):
    if text is None:
        return 0
    vowels = "aeiouyAEIOUY"
    count = 0
    prev_vowel = False
    for char in text:
        is_vowel = char in vowels
        if is_vowel and not prev_vowel:
            count += 1
        prev_vowel = is_vowel
    return count

syllable_udf = udf(compute_syllables, LongType())

df_fk = df_annotated \
    .withColumn("n_sentences", size(col("sentences"))) \
    .withColumn("n_syllables", syllable_udf(col("text"))) \
    .withColumn("flesch_kincaid", spark_round(
        206.835
        - 1.015 * (col("n_words") / col("n_sentences"))
        - 84.6  * (col("n_syllables") / col("n_words")), 2)) \
    .withColumn("kandel_moles", spark_round(
        207
        - 1.015 * (col("n_words") / col("n_sentences"))
        - 73.6  * (col("n_syllables") / col("n_words")), 2))

df_fk.select("author", "title", "movement", "flesch_kincaid", "kandel_moles").show()

+----------+------------------+--------+--------------+------------+
|    author|             title|movement|flesch_kincaid|kandel_moles|
+----------+------------------+--------+--------------+------------+
|    Balzac|       Pere_Goriot|romantic|         66.82|       83.39|
|    Balzac|   Eugenie_Grandet|romantic|         69.31|       85.27|
|     Dumas|      Monte_Cristo|romantic|         60.69|       77.95|
|  Stendhal|     Rouge_et_Noir|romantic|         61.09|       78.27|
|      Sand|           Indiana|romantic|         69.37|       86.04|
|  Bronte_C|         Jane_Eyre|romantic|         64.48|       81.16|
|  Bronte_E| Wuthering_Heights|romantic|         64.96|       81.62|
|     Scott|           Ivanhoe|romantic|         50.23|       67.54|
|  Flaubert|     Madame_Bovary| realist|         63.62|        80.4|
|      Zola|          Germinal| realist|         57.63|       75.32|
|      Zola|         Assommoir| realist|         65.66|       82.27|
|   Dickens|Great_Expectations| re

# Analyse Stylométrique

## Indices de lisibilité

Les indices de lisibilité mesurent quantitativement la difficulté de lecture  d'un texte. Nous calculons deux indices : **Flesch-Kincaid** (FK) et
**Kandel-Moles** (KM), tous deux basés sur la longueur des phrases et lacomplexité des mots.

### Flesch-Kincaid

La formule de Flesch-Kincaid est :

$$FK = 206.835 - 1.015 \times \frac{mots}{phrases} - 84.6 \times \frac{syllabes}{mots}$$

Un score élevé indique un texte facile à lire. Les scores varient généralement entre 0 (très difficile) et 100 (très facile).

In [ ]:
import plotly.express as px

df_plot = df_fk.select("author", "title", "movement", "flesch_kincaid", "kandel_moles").toPandas()

# Moyenne par auteur
df_avg = df_plot.groupby(["author", "movement"], as_index=False).agg(
    flesch_kincaid=("flesch_kincaid", "mean"),
    kandel_moles=("kandel_moles", "mean")
).round(2)

fig = px.bar(
    df_avg.sort_values("flesch_kincaid"),
    x="author",
    y="flesch_kincaid",
    color="movement",
    title="Indice de lisibilité Flesch-Kincaid par auteur (moyenne)",
    labels={
        "flesch_kincaid": "Score Flesch-Kincaid",
        "author": "Auteur",
        "movement": "Mouvement littéraire"
    },
    color_discrete_map={"romantic": "#e07b54", "realist": "#4a90d9"}
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
fig2 = px.bar(
    df_avg.sort_values("kandel_moles"),
    x="author",
    y="kandel_moles",
    color="movement",
    title="Indice de lisibilité Kandel-Moles par auteur (moyenne)",
    labels={
        "kandel_moles": "Score Kandel-Moles",
        "author": "Auteur",
        "movement": "Mouvement littéraire"
    },
    color_discrete_map={"romantic": "#e07b54", "realist": "#4a90d9"}
)

fig2.update_layout(xaxis_tickangle=-45)
fig2.show()

## Loi de Zipf

La loi de Zipf stipule que dans tout corpus textuel, la fréquence d'un mot est inversement proportionnelle à son rang. Autrement dit, le mot le plus fréquent apparaît environ 2 fois plus que le 2ème, 3 fois plus que le 3ème, et ainsi de suite.

Sur un graphique log-log (rang vs fréquence), cette relation donne une droite de pente -1.

Nous calculons les fréquences des mots pour chaque roman, puis nous traçons le graphique rang/fréquence pour vérifier si la loi de Zipf s'applique à notre corpus.

In [ ]:
from pyspark.sql.functions import explode, lower, regexp_replace, count
import plotly.express as px

df_one = df_annotated.filter(col("title") == "Pere_Goriot")

df_words = df_one.select(
    "title",
    explode("tokens").alias("token")
).select(
    "title",
    lower(col("token.result")).alias("word")
)

df_words = df_words.filter(col("word").rlike("^[a-z]+$"))

df_freq = df_words.groupBy("word") \
                  .agg(count("*").alias("freq")) \
                  .orderBy(col("freq").desc())

df_freq.show(20)

+----+----+
|word|freq|
+----+----+
| the|5315|
|  to|3052|
|  of|2872|
|   a|2620|
| and|2575|
|  in|1710|
|  he|1513|
|that|1481|
| you|1465|
|   i|1373|
| his|1149|
|  is|1137|
| for| 947|
| was| 937|
|  it| 922|
| her| 894|
|  as| 839|
| not| 701|
|  at| 698|
| had| 698|
+----+----+
only showing top 20 rows



In [ ]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

window = Window.orderBy(col("freq").desc())
df_zipf = df_freq.withColumn("rank", row_number().over(window))

df_zipf_pd = df_zipf.toPandas()

fig = px.scatter(
    df_zipf_pd.head(1000),
    x="rank",
    y="freq",
    log_x=True,
    log_y=True,
    title="Loi de Zipf — Père Goriot (Balzac)",
    labels={
        "rank": "Rang (log)",
        "freq": "Fréquence (log)"
    }
)

fig.show()

La courbe log-log obtenue pour *Père Goriot* confirme la loi de Zipf :
on observe une droite de pente négative quasi-parfaite, caractéristique de tout texte naturel. Les premiers rangs (mots très fréquents comme "the", "to", "of") s'écartent légèrement de la droite c'est normal et observé dans tous les corpus littéraires.

Nous reproduisons maintenant cette analyse sur l'ensemble des 14 romans pour comparer les distributions de fréquences entre auteurs et mouvements.

In [ ]:
from pyspark.sql.functions import explode, lower, count, row_number
from pyspark.sql.window import Window

# Pour tous les romans
df_words_all = df_annotated.select(
    "title", "author",
    explode("tokens").alias("token")
).select(
    "title", "author",
    lower(col("token.result")).alias("word")
).filter(col("word").rlike("^[a-z]+$"))

# Fréquences par roman
df_freq_all = df_words_all.groupBy("title", "author", "word") \
                          .agg(count("*").alias("freq"))

# Rang par roman
window = Window.partitionBy("title").orderBy(col("freq").desc())
df_zipf_all = df_freq_all.withColumn("rank", row_number().over(window))

# On garde les 500 premiers rangs par roman
df_zipf_pd = df_zipf_all.filter(col("rank") <= 500).toPandas()

fig = px.line(
    df_zipf_pd.sort_values("rank"),
    x="rank",
    y="freq",
    color="title",
    log_x=True,
    log_y=True,
    title="Loi de Zipf — tous les romans",
    labels={
        "rank": "Rang (log)",
        "freq": "Fréquence (log)",
        "title": "Roman"
    }
)

fig.show()

Le graphique confirme que la loi de Zipf s'applique à l'ensemble du corpus, toutes les courbes sont quasi-parallèles sur l'échelle log-log, ce qui est
caractéristique de tout texte naturel.

Deux observations :

- **Indiana** (Sand) se détache nettement en dessous des autres en raison
  de sa taille bien plus réduite (14 000 mots contre 100 000+ pour les autres)
- Les courbes sont très proches entre elles, la distribution des fréquences
  de mots est similaire pour tous les auteurs, indépendamment du mouvement
  littéraire (romantique ou réaliste)

## Fenêtres glissantes

Les indices de lisibilité calculés sur l'ensemble d'un roman donnent une valeur globale qui masque les variations internes du texte.
Pour observer comment le style évolue au fil du roman, nous calculons FK sur des fenêtres glissantes de taille fixe (500, 1000 et 2000 mots).

Plus la fenêtre est grande, plus l'indice est stable. Une fenêtre
petite révèle davantage de variations locales.

In [ ]:
from pyspark.sql.functions import posexplode, floor
import plotly.express as px

# On travaille sur un seul roman pour commencer : Pere Goriot
df_one = df_annotated.filter(col("title") == "Pere_Goriot")

# On explose les tokens avec leur position
df_tokens = df_one.select(
    "title",
    posexplode("tokens").alias("pos", "token")
).select(
    "title",
    col("pos"),
    col("token.result").alias("word")
)

df_tokens.show(5)

+-----------+---+------+
|      title|pos|  word|
+-----------+---+------+
|Pere_Goriot|  0| EBOOK|
|Pere_Goriot|  1|FATHER|
|Pere_Goriot|  2|GORIOT|
|Pere_Goriot|  3|   ***|
|Pere_Goriot|  4|FATHER|
+-----------+---+------+
only showing top 5 rows



In [ ]:
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import floor, count, sum as spark_sum

WINDOW_SIZE = 1000

def count_syllables_word(word):
    if word is None:
        return 0
    vowels = "aeiouyAEIOUY"
    count_s = 0
    prev_vowel = False
    for char in word.lower():
        is_vowel = char in vowels
        if is_vowel and not prev_vowel:
            count_s += 1
        prev_vowel = is_vowel
    return max(1, count_s)

syllable_word_udf = udf(count_syllables_word, IntegerType())

df_windows = df_tokens \
    .filter(col("word").rlike("^[a-zA-Z]+$")) \
    .withColumn("window_id", floor(col("pos") / WINDOW_SIZE)) \
    .withColumn("syllables", syllable_word_udf(col("word")))

df_agg = df_windows.groupBy("title", "window_id") \
    .agg(
        count("*").alias("n_words"),
        spark_sum("syllables").alias("n_syllables")
    )

df_sliding = df_agg \
    .withColumn("n_sentences_est", (col("n_words") / 15).cast("double")) \
    .withColumn("flesch_kincaid", spark_round(
        206.835
        - 1.015 * (col("n_words") / col("n_sentences_est"))
        - 84.6  * (col("n_syllables") / col("n_words")), 2
    ))

df_sliding.orderBy("window_id").show(10)

+-----------+---------+-------+-----------+------------------+--------------+
|      title|window_id|n_words|n_syllables|   n_sentences_est|flesch_kincaid|
+-----------+---------+-------+-----------+------------------+--------------+
|Pere_Goriot|        0|    864|       1336|              57.6|         60.79|
|Pere_Goriot|        1|    876|       1331|              58.4|         63.07|
|Pere_Goriot|        2|    862|       1343| 57.46666666666667|          59.8|
|Pere_Goriot|        3|    847|       1359| 56.46666666666667|         55.87|
|Pere_Goriot|        4|    872|       1362| 58.13333333333333|         59.47|
|Pere_Goriot|        5|    852|       1295|              56.8|         63.02|
|Pere_Goriot|        6|    887|       1432| 59.13333333333333|         55.03|
|Pere_Goriot|        7|    836|       1309|55.733333333333334|         59.14|
|Pere_Goriot|        8|    836|       1270|55.733333333333334|         63.09|
|Pere_Goriot|        9|    880|       1386|58.666666666666664|  

In [ ]:
df_sliding_pd = df_sliding.orderBy("window_id").toPandas()

fig = px.line(
    df_sliding_pd,
    x="window_id",
    y="flesch_kincaid",
    title="Indice FK sur fenêtres glissantes — Père Goriot (fenêtre = 1000 mots)",
    labels={
        "window_id": "Fenêtre (x1000 mots)",
        "flesch_kincaid": "Score Flesch-Kincaid"
    }
)

fig.add_hline(
    y=df_sliding_pd["flesch_kincaid"].mean(),
    line_dash="dash",
    line_color="red",
    annotation_text="Moyenne"
)

fig.show()

Le score FK de *Père Goriot* oscille entre 55 et 80 autour d'une moyenne stable de 67. La chute en fin de roman correspond probablement à des passages plus denses ou des chapitres de conclusion plus courts.

Nous reproduisons maintenant cette analyse sur l'ensemble des 14 romans pour comparer la stabilité du style entre auteurs et mouvements.

In [ ]:
# Sur tous les romans
df_tokens_all = df_annotated.select(
    "title", "author", "movement",
    posexplode("tokens").alias("pos", "token")
).select(
    "title", "author", "movement",
    col("pos"),
    col("token.result").alias("word")
)

df_windows_all = df_tokens_all \
    .filter(col("word").rlike("^[a-zA-Z]+$")) \
    .withColumn("window_id", floor(col("pos") / WINDOW_SIZE)) \
    .withColumn("syllables", syllable_word_udf(col("word")))

df_agg_all = df_windows_all.groupBy("title", "author", "movement", "window_id") \
    .agg(
        count("*").alias("n_words"),
        spark_sum("syllables").alias("n_syllables")
    )

df_sliding_all = df_agg_all \
    .withColumn("n_sentences_est", (col("n_words") / 15).cast("double")) \
    .withColumn("flesch_kincaid", spark_round(
        206.835
        - 1.015 * (col("n_words") / col("n_sentences_est"))
        - 84.6  * (col("n_syllables") / col("n_words")), 2
    ))

df_sliding_pd_all = df_sliding_all.orderBy("title", "window_id").toPandas()

fig = px.line(
    df_sliding_pd_all,
    x="window_id",
    y="flesch_kincaid",
    color="title",
    title="Indice FK sur fenêtres glissantes — tous les romans (fenêtre = 1000 mots)",
    labels={
        "window_id": "Fenêtre (x1000 mots)",
        "flesch_kincaid": "Score Flesch-Kincaid",
        "title": "Roman"
    }
)

fig.show()

Le graphique montre que tous les romans ont un comportement similaire, les scores FK oscillent entre 45 et 85 avec une moyenne autour de 65, indépendamment du mouvement littéraire.

Deux observations :
- Les indices sont **relativement stables** au fil des romans — le style d'un auteur ne change pas drastiquement d'un chapitre à l'autre
- Les variations locales sont plus importantes en début et fin de roman, probablement dues aux chapitres d'introduction et de conclusion qui ont des structures différentes de la narration principale

On passe maintenant à la segmentation **dialogue vs narration**.

## Fenêtres glissantes - comparaison des tailles

Un indice calculé sur l'ensemble d'un roman donne une valeur globale
qui masque les variations internes du texte. Les fenêtres glissantes
permettent d'observer comment le style évolue au fil du roman.

La question clé est : à partir de quelle taille de fenêtre l'indice
devient-il stable ? Nous comparons trois tailles : 500, 1000 et 2000 mots.

In [34]:
from pyspark.sql.functions import posexplode, floor, count, sum as spark_sum, col, lit
from pyspark.sql.functions import round as spark_round
import plotly.express as px
import pandas as pd

df_one = df_annotated.filter(col("title") == "Pere_Goriot")

df_tokens_one = df_one.select(
    "title",
    posexplode("tokens").alias("pos", "token")
).select(
    "title",
    col("pos"),
    col("token.result").alias("word")
).filter(col("word").rlike("^[a-zA-Z]+$")) \
 .withColumn("syllables", syllable_word_udf(col("word")))

all_windows = []

for window_size in [500, 1000, 2000]:
    df_w = df_tokens_one \
        .withColumn("window_id", floor(col("pos") / window_size)) \
        .groupBy("title", "window_id") \
        .agg(
            count("*").alias("n_words"),
            spark_sum("syllables").alias("n_syllables")
        ) \
        .withColumn("n_sentences_est", (col("n_words") / 15).cast("double")) \
        .withColumn("flesch_kincaid", spark_round(
            206.835
            - 1.015 * (col("n_words") / col("n_sentences_est"))
            - 84.6  * (col("n_syllables") / col("n_words")), 2
        )) \
        .withColumn("window_size", lit(window_size))

    all_windows.append(df_w.orderBy("window_id").toPandas())

df_all_windows = pd.concat(all_windows, ignore_index=True)
df_all_windows["window_size"] = df_all_windows["window_size"].astype(str) + " mots"

fig = px.line(
    df_all_windows,
    x="window_id",
    y="flesch_kincaid",
    color="window_size",
    title="Stabilite de l'indice FK selon la taille de fenetre - Pere Goriot",
    labels={
        "window_id": "Position dans le roman (numero de fenetre)",
        "flesch_kincaid": "Score Flesch-Kincaid",
        "window_size": "Taille de fenetre"
    }
)
fig.show()

stability = df_all_windows.groupby("window_size")["flesch_kincaid"].agg(
    ["mean", "std", "min", "max"]
).round(2)
stability.columns = ["Moyenne", "Ecart-type", "Min", "Max"]
print("\nStabilite de FK par taille de fenetre :")
print(stability.to_string())


Stabilite de FK par taille de fenetre :
             Moyenne  Ecart-type    Min    Max
window_size                                   
1000 mots      67.39        6.00  40.26  80.93
2000 mots      67.14        5.86  40.26  80.00
500 mots       67.40        6.84  35.84  81.84


### Analyse de la stabilité

Le graphique confirme l'intuition : plus la fenêtre est grande, plus
la courbe est lisse et stable.

Le tableau le confirme avec l'écart-type :
- 500 mots : écart-type de 6.84, la courbe est très bruitée,
  les variations locales du style sont visibles mais difficiles à interpréter
- 1000 mots : écart-type de 6.00, bon compromis entre sensibilité et lisibilité
- 2000 mots : écart-type de 5.86, courbe la plus stable mais les variations
  locales sont gommées

La moyenne reste identique (~67) quelle que soit la taille de fenêtre,
ce qui confirme que le style global de Balzac est cohérent tout au long
du roman. Seule la granularité de l'observation change.

Conclusion : une fenêtre de 1000 mots est le meilleur compromis pour
notre corpus. Elle capture les variations stylistiques sans être trop
sensible au bruit local.

## Dialogue vs Narration

La segmentation dialogue/narration permet de caractériser le style d'un auteur. Un roman avec beaucoup de dialogues aura un style plus vivant et direct, tandis qu'un roman narratif sera plus descriptif.

Nous détectons les dialogues comme les passages entre guillemets
(`"..."`) et calculons la proportion de dialogue par rapport à la
narration pour chaque roman.

In [ ]:
from pyspark.sql.functions import regexp_extract_all, size, col
from pyspark.sql.types import ArrayType, StringType, IntegerType

def extract_dialog(text):
    if text is None:
        return 0, 0
    import re
    # Guillemets typographiques " " (unicode) ET guillemets standards " "
    dialogs = re.findall(r'[\u201c"]([^\u201d"]*)[\u201d"]', text)
    dialogs += re.findall(r'(?m)^[—–-]\s(.+)$', text)
    dialog_chars = sum(len(d) for d in dialogs)
    total_chars = len(text)
    return int(dialog_chars), int(total_chars)

dialog_schema = StructType([
    StructField("dialog_chars", IntegerType()),
    StructField("total_chars",  IntegerType())
])
dialog_udf = udf(extract_dialog, dialog_schema)

df_dialog = df_annotated \
    .withColumn("dialog_stats", dialog_udf(col("text"))) \
    .withColumn("dialog_chars", col("dialog_stats.dialog_chars")) \
    .withColumn("total_chars", col("dialog_stats.total_chars")) \
    .withColumn("dialog_pct", spark_round(
        col("dialog_chars") / col("total_chars") * 100, 2)) \
    .withColumn("narration_pct", spark_round(
        100 - col("dialog_pct"), 2))

df_dialog.select("author", "title", "movement", "dialog_pct", "narration_pct").show()

+----------+------------------+--------+----------+-------------+
|    author|             title|movement|dialog_pct|narration_pct|
+----------+------------------+--------+----------+-------------+
|    Balzac|       Pere_Goriot|romantic|     47.06|        52.94|
|    Balzac|   Eugenie_Grandet|romantic|      34.4|         65.6|
|     Dumas|      Monte_Cristo|romantic|       0.0|        100.0|
|  Stendhal|     Rouge_et_Noir|romantic|       0.0|        100.0|
|      Sand|           Indiana|romantic|     25.53|        74.47|
|  Bronte_C|         Jane_Eyre|romantic|      41.8|         58.2|
|  Bronte_E| Wuthering_Heights|romantic|     44.35|        55.65|
|     Scott|           Ivanhoe|romantic|     44.44|        55.56|
|  Flaubert|     Madame_Bovary| realist|     17.86|        82.14|
|      Zola|          Germinal| realist|       0.0|        100.0|
|      Zola|         Assommoir| realist|     13.17|        86.83|
|   Dickens|Great_Expectations| realist|     27.93|        72.07|
|   Dicken

In [ ]:
import plotly.express as px
df_dialog_pd = df_dialog.select("author", "title", "movement", "dialog_pct", "narration_pct").toPandas()
df_dialog_avg = df_dialog_pd.groupby(["author", "movement"], as_index=False).agg(
    dialog_pct=("dialog_pct", "mean"),
    narration_pct=("narration_pct", "mean")
).round(2)

df_melted = df_dialog_avg.melt(
    id_vars=["author", "movement"],
    value_vars=["dialog_pct", "narration_pct"],
    var_name="type",
    value_name="percentage"
)

df_melted["type"] = df_melted["type"].map({
    "dialog_pct": "Dialogue",
    "narration_pct": "Narration"
})

fig = px.bar(
    df_melted.sort_values("author"),
    x="author",
    y="percentage",
    color="type",
    title="Proportion dialogue vs narration par auteur (moyenne)",
    labels={
        "percentage": "Pourcentage (%)",
        "author": "Auteur",
        "type": "Type"
    },
    color_discrete_map={"Dialogue": "#e07b54", "Narration": "#4a90d9"}
)

fig.update_layout(xaxis_tickangle=-45, barmode="stack")
fig.show()

### Détection des dialogues

La détection des dialogues s'est révélée plus complexe que prévu.
Une première approche avec des guillemets standards `"..."` ne donnait aucun résultat pour la majorité des romans. En inspectant le texte brut, nous avons découvert que Gutenberg utilise des guillemets typographiques unicode `\u201c` et `\u201d` (`"` et `"`) au lieu des guillemets ASCII standards.

Par ailleurs, certains romans français (Germinal, Rouge et Noir,
Monte Cristo) utilisent les tirets `—` pour les dialogues, convention typographique française. Ces romans affichent donc 0% de dialogue détecté, non pas parce qu'ils n'ont pas de dialogues, mais parce que leur format typographique diffère.

Enfin, certains auteurs ayant deux romans dans le corpus (Balzac,
Dickens, Zola), nous calculons la moyenne des proportions par auteur.

### Résultats
Les romantiques anglais (Brontë, Scott, Balzac) ont les proportions de dialogue les plus élevées (40-45%), tandis que les réalistes français (Zola, Flaubert, Maupassant) sont plus narratifs (moins de 20% de dialogue).

# Spark — Performance Analysis

## Comparaison des formats de fichiers

Spark peut lire et écrire des données dans plusieurs formats. Nous
comparons ici **CSV**, **JSON**, **Parquet** et **ORC** sur trois critères :

- **Temps d'écriture**: combien de secondes pour sauvegarder le DataFrame
- **Temps de lecture**: combien de secondes pour relire les données
- **Taille sur disque**: espace occupé en MB

CSV et JSON sont des formats texte lisibles par tous mais non optimisés
pour Spark. Parquet et ORC sont des formats orientés colonne conçus pour
les traitements distribués  ils compressent mieux et permettent à Spark
de ne lire que les colonnes nécessaires.

L'objectif est de justifier notre choix de Parquet comme format de
stockage intermédiaire tout au long du projet.

In [ ]:
import time
import shutil

BENCH_DIR = DATA_DIR / "benchmarks"
BENCH_DIR.mkdir(parents=True, exist_ok=True)

def benchmark_format(fmt, df):
    path = str(BENCH_DIR / fmt)
    if Path(path).exists():
        shutil.rmtree(path)

    # Ecriture
    start = time.perf_counter()
    if fmt == "csv":
        df.write.mode("overwrite").csv(path, header=True)
    elif fmt == "json":
        df.write.mode("overwrite").json(path)
    elif fmt == "parquet":
        df.write.mode("overwrite").parquet(path)
    elif fmt == "orc":
        df.write.mode("overwrite").orc(path)
    write_time = round(time.perf_counter() - start, 2)

    # Taille
    size_mb = round(sum(
        f.stat().st_size for f in Path(path).rglob("*") if f.is_file()
    ) / 1024 / 1024, 2)

    # Lecture
    start = time.perf_counter()
    if fmt == "csv":
        spark.read.csv(path, header=True).count()
    elif fmt == "json":
        spark.read.json(path).count()
    elif fmt == "parquet":
        spark.read.parquet(path).count()
    elif fmt == "orc":
        spark.read.orc(path).count()
    read_time = round(time.perf_counter() - start, 2)

    print(f"[{fmt}] write={write_time}s | read={read_time}s | size={size_mb}MB")
    return {"format": fmt, "write_seconds": write_time,
            "read_seconds": read_time, "size_mb": size_mb}

# On benchmark sur le corpus sans les colonnes complexes
df_bench = spark.read.parquet(str(DATA_DIR / "corpus_raw.parquet"))

results = []
for fmt in ["csv", "json", "parquet", "orc"]:
    results.append(benchmark_format(fmt, df_bench))

print("\nBenchmark terminé !")

[csv] write=1.12s | read=0.64s | size=10.03MB
[json] write=0.82s | read=0.53s | size=10.23MB
[parquet] write=0.82s | read=0.25s | size=6.36MB
[orc] write=1.09s | read=0.54s | size=6.38MB

Benchmark terminé !


In [ ]:
import pandas as pd

benchmark_pd = pd.DataFrame(results)

bench_long = benchmark_pd.melt(
    id_vars=["format"],
    value_vars=["write_seconds", "read_seconds", "size_mb"],
    var_name="mesure",
    value_name="valeur"
)

bench_long["mesure"] = bench_long["mesure"].map({
    "write_seconds": "Écriture (s)",
    "read_seconds": "Lecture (s)",
    "size_mb": "Taille (MB)"
})

fig = px.bar(
    bench_long,
    x="format",
    y="valeur",
    color="format",
    facet_col="mesure",
    title="Comparaison des formats — CSV, JSON, Parquet, ORC",
    labels={"valeur": "Valeur", "format": "Format"},
)

fig.update_yaxes(matches=None)
fig.show()

## Résultats

| Format  | Écriture (s) | Lecture (s) | Taille (MB) |
|---------|-------------|-------------|-------------|
| CSV     | 1.02        | 0.58        | 10.03       |
| JSON    | 0.69        | 0.59        | 10.23       |
| Parquet | 0.77        | 0.29        | 6.37        |
| ORC     | 0.92        | 0.46        | 6.39        |

**Parquet** est le meilleur format sur deux critères essentiels :

- **Lecture la plus rapide** (0.29s) — 2x plus rapide que CSV et JSON.
  C'est crucial car dans un pipeline Spark on lit les données bien plus
  souvent qu'on ne les écrit.
- **Taille la plus compacte** (6.37MB) — 37% plus léger que CSV grâce
  à la compression Snappy et au stockage orienté colonne.

**ORC** obtient des résultats similaires à Parquet, c'est un format
équivalent, historiquement lié à l'écosystème Hive/Hadoop.

**CSV et JSON** sont plus lents en lecture et occupent plus d'espace,
ils ne tirent pas parti des optimisations de Spark (pushdown de prédicats,
lecture par colonne).

**Conclusion** : nous utilisons Parquet comme format de stockage
intermédiaire tout au long du projet pour ses performances en lecture
et sa compacité.

## Cache et Persist

Par défaut Spark recalcule un DataFrame depuis le début à chaque action
(`.count()`, `.show()`, etc.). Le mécanisme de **cache** permet de stocker
un DataFrame en mémoire pour éviter ces recalculs.

Nous mesurons ici l'impact du cache sur notre corpus annoté  le DataFrame
le plus lourd du projet.

In [ ]:
import time

df_test = spark.read.parquet(str(DATA_DIR / "corpus_annotated.parquet"))

# Sans cache
start = time.perf_counter()
df_test.count()
t1 = round(time.perf_counter() - start, 2)

start = time.perf_counter()
df_test.count()
t2 = round(time.perf_counter() - start, 2)

print(f"Sans cache — 1er appel : {t1}s | 2ème appel : {t2}s")

# Avec cache
df_cached = spark.read.parquet(str(DATA_DIR / "corpus_annotated.parquet")).cache()

start = time.perf_counter()
df_cached.count()  # déclenche le cache
t3 = round(time.perf_counter() - start, 2)

start = time.perf_counter()
df_cached.count()  # lit depuis la mémoire
t4 = round(time.perf_counter() - start, 2)

print(f"Avec cache  — 1er appel : {t3}s | 2ème appel : {t4}s")
print(f"\nGain : {round((t2 - t4) / t2 * 100, 1)}% plus rapide avec cache")

df_cached.unpersist()

Sans cache — 1er appel : 0.2s | 2ème appel : 0.17s
Avec cache  — 1er appel : 1.39s | 2ème appel : 0.1s

Gain : 41.2% plus rapide avec cache


DataFrame[author: string, title: string, movement: string, lang: string, text: string, n_chars: bigint, n_words: bigint, document: array<struct<annotatorType:string,begin:int,end:int,result:string,metadata:map<string,string>,embeddings:array<float>>>, sentences: array<struct<annotatorType:string,begin:int,end:int,result:string,metadata:map<string,string>,embeddings:array<float>>>, tokens: array<struct<annotatorType:string,begin:int,end:int,result:string,metadata:map<string,string>,embeddings:array<float>>>]

## Résultats Cache

| Appel | Sans cache | Avec cache |
|-------|-----------|------------|
| 1er   | 0.22s     | 1.48s      |
| 2ème  | 0.21s     | 0.07s      |

**Observations :**

- Le **1er appel avec cache** est plus lent (1.48s vs 0.22s), c'est normal,
  Spark doit lire le fichier ET le stocker en mémoire en même temps.
- Le **2ème appel avec cache** est 3x plus rapide (0.07s vs 0.21s), Spark
  lit directement depuis la mémoire sans toucher au disque.
- Le **gain de 66.7%** devient significatif quand on appelle le même
  DataFrame de nombreuses fois comme dans notre pipeline stylométrique
  où `df_annotated` est utilisé pour FK, KM, Zipf, fenêtres glissantes
  et dialogue/narration.

**Conclusion** : le cache est particulièrement utile pour les DataFrames
lourds réutilisés plusieurs fois. Dans notre cas, mettre `df_annotated`
en cache au début de la stylométrie aurait réduit le temps total du pipeline.

## Persist avec différents niveaux de stockage

`cache()` est un raccourci pour `persist(MEMORY_AND_DISK)`.
Spark propose en réalité plusieurs niveaux de stockage avec des
compromis différents entre vitesse et consommation mémoire.

Nous comparons ici trois niveaux sur `df_annotated` :
- `MEMORY_ONLY` : stockage uniquement en RAM, les partitions qui ne rentrent pas sont recalculées
- `MEMORY_AND_DISK` : ce que fait cache() par défaut, déborde sur disque si nécessaire
- `DISK_ONLY` : stockage uniquement sur disque, libère la RAM pour d'autres DataFrames

In [29]:
from pyspark import StorageLevel
import time

df_base = spark.read.parquet(str(DATA_DIR / "corpus_annotated.parquet"))

levels = {
    "MEMORY_ONLY":     StorageLevel.MEMORY_ONLY,
    "MEMORY_AND_DISK": StorageLevel.MEMORY_AND_DISK,
    "DISK_ONLY":       StorageLevel.DISK_ONLY,
}

persist_results = []

for name, level in levels.items():
    df_p = df_base.persist(level)

    start = time.perf_counter()
    df_p.count()
    t1 = round(time.perf_counter() - start, 2)

    start = time.perf_counter()
    df_p.count()
    t2 = round(time.perf_counter() - start, 2)

    df_p.unpersist()
    persist_results.append({"niveau": name, "1er_appel_s": t1, "2eme_appel_s": t2})
    print(f"[{name}]  1er={t1}s | 2eme={t2}s")

import pandas as pd
print(pd.DataFrame(persist_results).to_string(index=False))

[MEMORY_ONLY]  1er=1.3s | 2eme=0.2s
[MEMORY_AND_DISK]  1er=2.26s | 2eme=0.31s
[DISK_ONLY]  1er=1.41s | 2eme=0.23s
         niveau  1er_appel_s  2eme_appel_s
    MEMORY_ONLY         1.30          0.20
MEMORY_AND_DISK         2.26          0.31
      DISK_ONLY         1.41          0.23


### Analyse des résultats

- `MEMORY_ONLY` est le plus rapide en lecture (0.20s) car tout est en RAM,
  aucun accès disque. Sur Colab notre corpus annoté rentre entièrement en mémoire,
  donc aucune partition n'est recalculée.

- `MEMORY_AND_DISK` est plus lent (0.31s) car Spark vérifie d'abord la mémoire
  puis le disque. Le surcoût de gestion est visible même quand tout tient en RAM.

- `DISK_ONLY` est intermédiaire (0.23s), plus lent que MEMORY_ONLY mais plus
  rapide que MEMORY_AND_DISK car il n'y a pas de double vérification mémoire/disque.

Sur un cluster avec moins de mémoire disponible, `MEMORY_AND_DISK` serait
le meilleur choix car il évite les recalculs quand la RAM est pleine.
Dans notre cas Colab avec 14 romans, `MEMORY_ONLY` est suffisant.

## Identification des shuffles

Spark UI tourne sur `localhost:4040` et n'est pas accessible directement
depuis Colab. Nous utilisons à la place `df.explain()` qui affiche le
plan d'exécution de Spark et permet d'identifier les opérations de shuffle.

Un shuffle se produit quand Spark doit redistribuer les données entre
les partitions, c'est l'opération la plus coûteuse du pipeline. Dans
notre projet les shuffles critiques sont provoqués par les `groupBy`
et `orderBy` du calcul Zipf et des fenêtres glissantes.

In [ ]:
# Plan d'exécution du calcul Zipf — identifie les shuffles
df_words_all = df_annotated.select(
    "title",
    explode("tokens").alias("token")
).select(
    "title",
    lower(col("token.result")).alias("word")
).filter(col("word").rlike("^[a-z]+$"))

df_freq_all = df_words_all.groupBy("title", "word") \
                          .agg(count("*").alias("freq"))

print("=== Plan d'exécution Zipf ===")
df_freq_all.explain()

=== Plan d'exécution Zipf ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[title#1, word#1381], functions=[count(1)])
   +- Exchange hashpartitioning(title#1, word#1381, 200), ENSURE_REQUIREMENTS, [plan_id=1448]
      +- HashAggregate(keys=[title#1, word#1381], functions=[partial_count(1)])
         +- Project [title#1, lower(token#1378.result) AS word#1381]
            +- Filter (isnotnull(token#1378.result) AND RLIKE(lower(token#1378.result), ^[a-z]+$))
               +- Generate explode(tokens#109), [title#1], false, [token#1378]
                  +- Project [title#1, UDF(array(UDF(array(UDF(text#4))))) AS tokens#109]
                     +- Filter ((size(UDF(array(UDF(array(UDF(text#4))))), true) > 0) AND isnotnull(UDF(array(UDF(array(UDF(text#4)))))))
                        +- Scan ExistingRDD[author#0,title#1,movement#2,lang#3,text#4,n_chars#5L,n_words#6L]




### Analyse du plan d'exécution

Le plan révèle un shuffle critique à la ligne : Exchange hashpartitioning(title, word, 200)

Ce shuffle se produit lors du `groupBy("title", "word")` Spark doit
rassembler tous les tokens identiques qui sont éparpillés dans les
différentes partitions pour les compter. C'est l'opération la plus
coûteuse du pipeline Zipf.

Le plan se lit de bas en haut :

1. **Scan**: lecture du Parquet
2. **Generate explode**: explosion des tokens en lignes séparées
3. **HashAggregate partiel**: comptage local par partition
4. **Exchange hashpartitioning**: ← **SHUFFLE** redistribution des données
5. **HashAggregate final**: comptage global

Le `200` dans `hashpartitioning(..., 200)` indique le nombre de partitions
après shuffle c'est la valeur par défaut de Spark
(`spark.sql.shuffle.partitions`). Pour notre corpus de 14 romans c'est
excessif on pourrait le réduire à 8 ou 16 pour éviter de créer trop
de petites partitions vides.

### Mitigation des shuffles

Pour réduire l'impact du shuffle, nous diminuons le nombre de partitions
de 200 (valeur par défaut) à 8, plus adapté à notre petit corpus de
14 romans.

In [ ]:
# Avant optimisation
start = time.perf_counter()
df_freq_all.count()
t_before = round(time.perf_counter() - start, 2)
print(f"Sans optimisation : {t_before}s")

# Après optimisation
spark.conf.set("spark.sql.shuffle.partitions", "8")

start = time.perf_counter()
df_freq_all.count()
t_after = round(time.perf_counter() - start, 2)
print(f"Avec optimisation (8 partitions) : {t_after}s")

print(f"Gain : {round((t_before - t_after) / t_before * 100, 1)}%")

# Remettre la valeur par défaut
spark.conf.set("spark.sql.shuffle.partitions", "200")

Sans optimisation : 244.88s
Avec optimisation (8 partitions) : 235.64s
Gain : 3.8%


### Résultats

| Stratégie | Temps |
|-----------|-------|
| 200 partitions (défaut) | 318.99s |
| 8 partitions (optimisé) | 318.89s |

Le gain est négligeable sur notre corpus de 14 romans. Avec un corpus
plus grand des centaines de romans par exemple la réduction du
nombre de partitions aurait probablement un impact plus visible, car
Spark passerait moins de temps à gérer des partitions vides.

## Mitigation des shuffles — stratégies supplémentaires

Nous avons vu que réduire `shuffle.partitions` de 200 à 8 n'apporte
qu'un gain négligeable sur notre corpus. Deux autres stratégies existent :

- `repartition()` avant un `groupBy` : si on redistribue les données
  par clé avant l'agrégation, Spark évite un shuffle supplémentaire
  car les données sont déjà groupées correctement.

- `broadcast join` : quand on joint un grand DataFrame avec un petit,
  Spark peut envoyer le petit DataFrame à chaque executor plutôt que
  de shuffler le grand. Ici nos métadonnées (14 lignes) sont le candidat idéal.

In [30]:
from pyspark.sql.functions import broadcast, explode, lower, count, col
import time

df_words_base = df_annotated.select(
    "title",
    explode("tokens").alias("token")
).select(
    "title",
    lower(col("token.result")).alias("word")
).filter(col("word").rlike("^[a-z]+$"))

# Sans repartition
start = time.perf_counter()
df_words_base.groupBy("title", "word").agg(count("*").alias("freq")).count()
t_no_repart = round(time.perf_counter() - start, 2)

# Avec repartition par title avant groupBy
start = time.perf_counter()
df_words_base \
    .repartition(14, "title") \
    .groupBy("title", "word").agg(count("*").alias("freq")).count()
t_repart = round(time.perf_counter() - start, 2)

print(f"Sans repartition      : {t_no_repart}s")
print(f"Avec repartition(14)  : {t_repart}s")
print(f"Gain : {round((t_no_repart - t_repart) / t_no_repart * 100, 1)}%")

Sans repartition      : 1.81s
Avec repartition(14)  : 2.31s
Gain : -27.6%


### Résultats repartition

La repartition est plus lente (-27.6%) sur notre corpus.

Sur 14 romans, les données sont déjà peu nombreuses et Spark les traite
rapidement sans repartitionner. Ajouter un repartition force un shuffle
supplémentaire avant le groupBy, ce qui coûte plus cher que ce qu'il économise.

Cette stratégie devient intéressante sur des corpus beaucoup plus grands
où les partitions sont très déséquilibrées, par exemple si certains romans
font 10x la taille des autres, repartitionner par title équilibrerait la charge
entre les executors.

In [31]:
import pandas as pd

meta = pd.DataFrame([
    {"title": b["title"], "movement": b["movement"], "lang": b["lang"]}
    for b in CORPUS
])
df_meta = spark.createDataFrame(meta)

# Sans broadcast
start = time.perf_counter()
df_annotated.join(df_meta, on="title").count()
t_no_broadcast = round(time.perf_counter() - start, 2)

# Avec broadcast
start = time.perf_counter()
df_annotated.join(broadcast(df_meta), on="title").count()
t_broadcast = round(time.perf_counter() - start, 2)

print(f"Join sans broadcast : {t_no_broadcast}s")
print(f"Join avec broadcast : {t_broadcast}s")
print(f"Gain : {round((t_no_broadcast - t_broadcast) / t_no_broadcast * 100, 1)}%")

print("\n=== Plan avec broadcast ===")
df_annotated.join(broadcast(df_meta), on="title").explain()

Join sans broadcast : 1.62s
Join avec broadcast : 1.04s
Gain : 35.8%

=== Plan avec broadcast ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [title#1411, author#1410, movement#1412, lang#1413, text#1414, n_chars#1415L, n_words#1416L, document#1417, sentences#1418, tokens#1419, movement#5000, lang#5001]
   +- BroadcastHashJoin [title#1411], [title#4999], Inner, BuildRight, false
      :- Filter isnotnull(title#1411)
      :  +- FileScan parquet [author#1410,title#1411,movement#1412,lang#1413,text#1414,n_chars#1415L,n_words#1416L,document#1417,sentences#1418,tokens#1419] Batched: true, DataFilters: [isnotnull(title#1411)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/drive/MyDrive/projet_BD/DATA/corpus_annotated.parquet], PartitionFilters: [], PushedFilters: [IsNotNull(title)], ReadSchema: struct<author:string,title:string,movement:string,lang:string,text:string,n_chars:bigint,n_words:...
      +- BroadcastExchange HashedRelationBroadcastMod

### Résultats broadcast join

Le broadcast join apporte un gain de 35.8% (1.62s -> 1.04s).

Le plan d'exécution confirme que Spark utilise un `BroadcastHashJoin` :
au lieu de shuffler `df_annotated` (14 romans, colonnes NLP lourdes),
Spark envoie `df_meta` (14 lignes) à chaque executor via un `BroadcastExchange`.
Le grand DataFrame ne bouge pas, seul le petit est distribué.

C'est la stratégie de mitigation la plus efficace de notre projet,
bien plus que la réduction de partitions ou la repartition.
Le broadcast est particulièrement adapté quand un des deux DataFrames
est petit et connu à l'avance, ce qui est toujours le cas pour nos métadonnées.

In [32]:
import pandas as pd

pd.DataFrame([
    {"Strategie": "200 partitions (defaut)",  "Temps (s)": 318.99, "Gain": "—"},
    {"Strategie": "8 partitions",              "Temps (s)": 318.89, "Gain": "~0%"},
    {"Strategie": "repartition(14, 'title')",  "Temps (s)": t_repart,       "Gain": f"{round((t_no_repart - t_repart)/t_no_repart*100,1)}%"},
    {"Strategie": "broadcast join",            "Temps (s)": t_broadcast,    "Gain": f"{round((t_no_broadcast - t_broadcast)/t_no_broadcast*100,1)}%"},
]).to_string(index=False)

"               Strategie  Temps (s)   Gain\n 200 partitions (defaut)     318.99      —\n            8 partitions     318.89    ~0%\nrepartition(14, 'title')       2.31 -27.6%\n          broadcast join       1.04  35.8%"